# Generate HFER Across Layers

This Generate notebook computes and caches the per-layer high-frequency energy ratio (HFER) consumed by `Visualize-diagnostics-across-layers.ipynb`. The companion Visualize notebook compares layer-specific graph-spectral diagnostics across all 28 layers at a fixed context length of $N = 2000$ tokens for the main numerical input families using Llama-3.2-3B, providing the across-layer comparison presented under **Robustness and Layerwise Consistency of Graph-Spectral Diagnostics** in the appendix.

Set imports, constants, relative output paths, fixed-context configuration, and random seeds.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import random
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Seed data generation and model inference
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Model and relative cache configuration
MODEL_NAME = "meta-llama/Llama-3.2-3B"
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = CACHE_DIR / "hfer_per_layer.npz"

EXPECTED_NUM_INPUT_FAMILIES = 10
NUM_ENSEMBLE_RUNS = 20
EXPECTED_NUM_LAYERS = 28

# Fixed context: 1,000 numeric states, each followed by a comma
CONTEXT_STATE_TOKENS = 1000
CONTEXT_COMMA_TOKENS = 1000
FIXED_CONTENT_TOKENS = 2000
TOTAL_SEQUENCE_STATES = 1001

hfer_fixed_state_tokens = CONTEXT_STATE_TOKENS
hfer_fixed_comma_tokens = CONTEXT_COMMA_TOKENS
hfer_fixed_content_tokens = FIXED_CONTENT_TOKENS
hfer_drop_bos = True
hfer_renormalize_after_drop = True
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
hfer_head_weighting = "mass"
hfer_tokenization_add_special_tokens = True

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
COMMA_TOKEN_IDS = tokenizer(",", add_special_tokens=False).input_ids
if len(COMMA_TOKEN_IDS) != 1:
    raise ValueError(f"Expected comma to tokenize as one token; got ids {COMMA_TOKEN_IDS}.")
COMMA_TOKEN_ID = int(COMMA_TOKEN_IDS[0])

Define the sequence generators and build the fixed-length ensemble used by the per-layer sweep.

In [ ]:
# Lorenz and logistic-map regimes, quantization, and input-family order
def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0

lorenz_dt = 100 / 999
t_eval = np.arange(TOTAL_SEQUENCE_STATES, dtype=float) * lorenz_dt
t_span = (float(t_eval[0]), float(t_eval[-1]))

periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
hfer_ensemble_periodic_rs = [3.2, 3.5, 3.55, 3.566, 3.5692, 3.5698]
hfer_ensemble_chaotic_rs = [3.9, 4.0]
logistic_rs = hfer_ensemble_periodic_rs + hfer_ensemble_chaotic_rs
logistic_expected_periods = {r: periodic_logistic_periods.get(r) for r in logistic_rs}
logistic_long_n = 20000
logistic_settle_tol = 1e-10

hfer_ensemble_data_order = [
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
]


def logistic_sequence(r, x0, n):
    x = np.empty(int(n), dtype=float)
    x[0] = float(x0)
    for i in range(1, int(n)):
        x[i] = float(r) * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def quantize_lorenz_x_values(x_values):
    x_values = np.asarray(x_values, dtype=float)
    x_min_rep = float(np.min(x_values))
    x_max_rep = float(np.max(x_values))
    if x_max_rep == x_min_rep:
        return np.full(x_values.shape[0], 500, dtype=int)
    return np.round(50 + (x_values - x_min_rep) / (x_max_rep - x_min_rep) * 900).astype(int)


def build_burned_in_logistic_window_ensemble(
    r,
    x0,
    n_window,
    expected_period=None,
    n_long=20000,
    tol=1e-10,
    phase_offset=0,
):
    if expected_period is None:
        vals = logistic_sequence(r, x0, n_window)
        return np.asarray(vals, dtype=float), 0, 0, False

    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + int(n_window) > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states."
        )

    vals = np.asarray(x_long[start_idx:start_idx + int(n_window)], dtype=float)
    return vals, int(t_settle), int(phase_offset), True

In [ ]:
# Share each replicate's sampled initial condition across input families
hfer_ensemble_R = NUM_ENSEMBLE_RUNS
hfer_ensemble_seed = seed
hfer_periodic_random_phase = True

hfer_ensemble_rng = np.random.default_rng(hfer_ensemble_seed)
x0_samples = hfer_ensemble_rng.uniform(0.3, 0.7, size=hfer_ensemble_R)
hfer_ensemble_rep_ids = list(range(hfer_ensemble_R))
hfer_ensemble_phase_rng = np.random.default_rng(hfer_ensemble_seed + 1)

hfer_ensemble_inputs = {}
constant_tokens = np.empty(hfer_ensemble_R, dtype=np.int64)
lorenz_initial_x0 = np.empty(hfer_ensemble_R, dtype=np.float64)

for rep_id, x0_rep in zip(hfer_ensemble_rep_ids, x0_samples):
    x0_rep = float(x0_rep)
    rep_inputs = {}
    lorenz_x0_rep = float(-8.0 + (x0_rep - 0.5))
    lorenz_y0_rep = 7.0
    lorenz_z0_rep = 27.0
    constant_token = int(np.round(50 + 900 * x0_rep))

    constant_tokens[int(rep_id)] = constant_token
    lorenz_initial_x0[int(rep_id)] = lorenz_x0_rep
    rep_inputs["Constant sampled"] = {
        "sequence": np.full(TOTAL_SEQUENCE_STATES, constant_token, dtype=int),
        "metadata": {"input_class": "constant", "x0_rep": x0_rep, "constant_token": constant_token},
    }

    for r in hfer_ensemble_periodic_rs:
        expected_period = logistic_expected_periods[r]
        if expected_period is None:
            raise ValueError(f"Expected a periodic window for r={r}, but expected_period is None.")

        phase_offset = (
            int(hfer_ensemble_phase_rng.integers(0, expected_period))
            if hfer_periodic_random_phase
            else 0
        )
        vals, burn_in_start_index, used_phase_offset, used_burn_in = build_burned_in_logistic_window_ensemble(
            r,
            x0=x0_rep,
            n_window=TOTAL_SEQUENCE_STATES,
            expected_period=expected_period,
            n_long=logistic_long_n,
            tol=logistic_settle_tol,
            phase_offset=phase_offset,
        )
        data_name = f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": quantize_logistic_window(vals),
            "metadata": {
                "input_class": "periodic",
                "x0_rep": x0_rep,
                "r": float(r),
                "expected_period": int(expected_period),
                "burn_in_start_index": int(burn_in_start_index),
                "phase_offset": int(used_phase_offset),
                "used_burn_in": bool(used_burn_in),
            },
        }

    for r in hfer_ensemble_chaotic_rs:
        vals = logistic_sequence(r, x0_rep, TOTAL_SEQUENCE_STATES)
        data_name = f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": quantize_logistic_window(vals),
            "metadata": {
                "input_class": "chaotic",
                "x0_rep": x0_rep,
                "r": float(r),
                "burn_in_start_index": 0,
                "phase_offset": 0,
                "used_burn_in": False,
            },
        }

    sol_rep = solve_ivp(
        lorenz,
        t_span,
        [lorenz_x0_rep, lorenz_y0_rep, lorenz_z0_rep],
        args=(sigma, rho, beta),
        t_eval=t_eval,
    )
    if not sol_rep.success:
        raise RuntimeError(f"Lorenz solve failed for rep_id={rep_id}: {sol_rep.message}")
    if sol_rep.y.shape[-1] != TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Expected {TOTAL_SEQUENCE_STATES} Lorenz samples for rep_id={rep_id}, got {sol_rep.y.shape[-1]}."
        )
    rep_inputs["Lorenz"] = {
        "sequence": quantize_lorenz_x_values(sol_rep.y[0]),
        "metadata": {
            "input_class": "lorenz",
            "x0_rep": x0_rep,
            "lorenz_x0_rep": lorenz_x0_rep,
            "lorenz_y0_rep": lorenz_y0_rep,
            "lorenz_z0_rep": lorenz_z0_rep,
        },
    }

    for data_name in hfer_ensemble_data_order:
        sequence = np.asarray(rep_inputs[data_name]["sequence"], dtype=int)
        if sequence.shape[0] != TOTAL_SEQUENCE_STATES:
            raise ValueError(
                f"{data_name}, rep_id={rep_id}: expected {TOTAL_SEQUENCE_STATES} states, got {sequence.shape[0]}."
            )
        if not np.all((0 <= sequence) & (sequence <= 999)):
            raise ValueError(f"{data_name}, rep_id={rep_id}: quantized states must be in [0, 999].")

    hfer_ensemble_inputs[int(rep_id)] = rep_inputs

print("Built ensemble inputs.")
print(f"  input families: {len(hfer_ensemble_data_order)}")
print(f"  ensemble runs: {len(hfer_ensemble_inputs)}")
print(f"  states per sequence: {TOTAL_SEQUENCE_STATES}")
print("  Constant sampled token convention: int(round(50 + 900 * x0_rep)).")
print("  Periodic logistic inputs use burn-in plus optional random phase.")
print("  Chaotic logistic inputs use sampled initial conditions without burn-in.")
print("  Lorenz varies only in x0 via -8 + (x0_rep - 0.5), with y0=7 and z0=27 fixed.")

Compute HFER from same-index attention and hidden-state layers after token alignment.

In [ ]:
# Token alignment, graph construction, and HFER utilities
def get_token_start_index(input_ids, bos_token_id=None, drop_bos=True):
    if not drop_bos:
        return 0
    if input_ids is None or bos_token_id is None:
        return 0

    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])

    return 1 if first_id == int(bos_token_id) else 0


def _resolve_hidden_state_layer_index(hidden_states, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(hidden_states) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(hidden_states):
        raise ValueError(
            f"Invalid hidden-state layer_index={layer_index} for {len(hidden_states)} hidden-state tensors"
        )
    return int(resolved_layer_index)


def _resolve_attention_layer_index(attentions, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(attentions) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(attentions):
        raise ValueError(
            f"Invalid attention layer_index={layer_index} for {len(attentions)} attention tensors"
        )
    return int(resolved_layer_index)


def compute_hfer_from_attn_and_hidden(
    attentions,
    hidden_states,
    input_ids=None,
    bos_token_id=None,
    k_cutoff=None,
    head_weighting="mass",
    drop_bos=True,
    renormalize_after_drop=True,
    attention_layer_index=-1,
    signal_layer_index=-2,
):
    token_start = get_token_start_index(
        input_ids,
        bos_token_id=bos_token_id,
        drop_bos=drop_bos,
    )

    resolved_attention_layer_index = _resolve_attention_layer_index(attentions, attention_layer_index)
    resolved_signal_layer_index = _resolve_hidden_state_layer_index(hidden_states, signal_layer_index)

    # Remove the leading BOS before constructing the token graph
    A = attentions[resolved_attention_layer_index][0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len <= 0:
        return {
            "hfer": float("nan"),
            "seq_len": seq_len,
            "k_cutoff": 0,
            "token_start": int(token_start),
            "resolved_attention_layer_index": int(resolved_attention_layer_index),
            "resolved_signal_layer_index": int(resolved_signal_layer_index),
        }

    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    if k_cutoff is None:
        k_cutoff = seq_len // 2

    # Symmetrize attention heads and combine them into one weighted graph
    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    D = torch.diag(Wbar.sum(dim=1))
    L = D - Wbar
    _, U = torch.linalg.eigh(L)

    X = hidden_states[resolved_signal_layer_index][0].float()
    X = X[token_start:, :].to(U.device)
    if X.shape[0] != seq_len:
        raise ValueError(
            f"Signal/token length mismatch after BOS handling: X has {X.shape[0]} rows, graph has {seq_len} nodes."
        )

    # Project hidden states onto graph Fourier modes and measure upper-half energy
    X_hat = U.T @ X
    energy = (X_hat ** 2).sum(dim=1)
    total = energy.sum().item()
    hfer_val = float("nan") if total == 0 else (energy[k_cutoff:].sum() / total).item()

    return {
        "hfer": float(hfer_val),
        "seq_len": int(seq_len),
        "k_cutoff": int(k_cutoff),
        "token_start": int(token_start),
        "resolved_attention_layer_index": int(resolved_attention_layer_index),
        "resolved_signal_layer_index": int(resolved_signal_layer_index),
    }


def compute_hfer_per_layer_from_attn_and_hidden(
    attentions,
    hidden_states,
    input_ids=None,
    bos_token_id=None,
    k_cutoff=None,
    head_weighting="mass",
    drop_bos=True,
    renormalize_after_drop=True,
):
    num_layers = len(attentions)
    if len(hidden_states) < num_layers:
        raise ValueError(
            f"Expected at least {num_layers} hidden-state tensors for {num_layers} attention layers; "
            f"got {len(hidden_states)}."
        )

    layer_results = []
    for layer_idx in range(num_layers):
        hfer_bundle = compute_hfer_from_attn_and_hidden(
            attentions,
            hidden_states,
            input_ids=input_ids,
            bos_token_id=bos_token_id,
            k_cutoff=k_cutoff,
            head_weighting=head_weighting,
            drop_bos=drop_bos,
            renormalize_after_drop=renormalize_after_drop,
            attention_layer_index=layer_idx,
            signal_layer_index=layer_idx,
        )
        hfer_bundle["layer_idx"] = int(layer_idx)
        hfer_bundle["layer_number"] = int(layer_idx + 1)
        layer_results.append(hfer_bundle)

    return layer_results


def _build_fixed_context_token_input_sequence(sequence):
    sequence = np.asarray(sequence, dtype=int)
    if sequence.size < TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Need at least {TOTAL_SEQUENCE_STATES} states, but sequence has {sequence.size}."
        )

    context_window = sequence[:CONTEXT_STATE_TOKENS]
    held_out_state = int(sequence[CONTEXT_STATE_TOKENS])
    if context_window.shape[0] != CONTEXT_STATE_TOKENS:
        raise ValueError(f"Expected {CONTEXT_STATE_TOKENS} context states.")

    input_sequence = ",".join([f"{int(x):03d}" for x in context_window]) + ","
    counts = {
        "state_tokens": int(context_window.size),
        "comma_tokens": int(input_sequence.count(",")),
        "content_tokens": int(context_window.size + input_sequence.count(",")),
        "total_sequence_states": int(sequence.size),
    }
    if not input_sequence.endswith(","):
        raise ValueError("Fixed-2000 HFER prompt must end with a comma.")
    if counts["state_tokens"] != CONTEXT_STATE_TOKENS:
        raise ValueError(f"Expected {CONTEXT_STATE_TOKENS} state tokens, got {counts['state_tokens']}.")
    if counts["comma_tokens"] != CONTEXT_COMMA_TOKENS:
        raise ValueError(f"Expected {CONTEXT_COMMA_TOKENS} comma tokens, got {counts['comma_tokens']}.")
    if counts["content_tokens"] != FIXED_CONTENT_TOKENS:
        raise ValueError(f"Expected {FIXED_CONTENT_TOKENS} content tokens, got {counts['content_tokens']}.")

    return input_sequence, counts, held_out_state


def _post_bos_token_count(input_ids, bos_token_id=None, drop_bos=True):
    token_start = get_token_start_index(
        input_ids,
        bos_token_id=bos_token_id,
        drop_bos=drop_bos,
    )
    if isinstance(input_ids, torch.Tensor):
        total_tokens = int(input_ids.reshape(-1).shape[0])
    else:
        total_tokens = int(np.asarray(input_ids).reshape(-1).shape[0])
    return int(total_tokens - token_start), int(token_start)





def _validate_prompt_tokenization(input_ids, data_name, rep_id):
    post_bos_token_count, token_start = _post_bos_token_count(
        input_ids,
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=hfer_drop_bos,
    )
    input_ids_flat = input_ids.reshape(-1)
    post_bos_token_ids = input_ids_flat[token_start:]
    if post_bos_token_count != FIXED_CONTENT_TOKENS:
        raise ValueError(
            f"Expected {FIXED_CONTENT_TOKENS} post-BOS tokens for {data_name}, rep_id={rep_id}; "
            f"got {post_bos_token_count}."
        )
    if int(post_bos_token_ids[-1].item()) != COMMA_TOKEN_ID:
        decoded_final = tokenizer.decode(
            [int(post_bos_token_ids[-1].item())],
            clean_up_tokenization_spaces=False,
        )
        raise ValueError(
            f"Expected final post-BOS token to be comma for {data_name}, rep_id={rep_id}; "
            f"got token_id={int(post_bos_token_ids[-1].item())}, decoded={decoded_final!r}."
        )

    return int(post_bos_token_count), int(token_start), int(post_bos_token_ids[-1].item())


print("HFER utilities ready.")
print("Layer pairing: attentions[layer_idx] with hidden_states[layer_idx].")
print("High-frequency cutoff remains the default k_cutoff = seq_len // 2.")

Load the model with eager attention and capture layer/device metadata.

In [ ]:
# Use eager attention to return full attention tensors
print("Loading model with eager attention implementation...")
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

hfer_num_layers = int(model_attn.config.num_hidden_layers)
if hfer_num_layers != EXPECTED_NUM_LAYERS:
    raise ValueError(f"Expected model to have {EXPECTED_NUM_LAYERS} layers, got {hfer_num_layers}.")

layer_numbers = np.arange(1, hfer_num_layers + 1, dtype=np.int64)
if not np.array_equal(layer_numbers, np.arange(1, EXPECTED_NUM_LAYERS + 1, dtype=np.int64)):
    raise ValueError("Layer-number metadata must be 1..28.")

print(
    "Model loaded: "
    f"{hfer_num_layers} layers, "
    f"{model_attn.config.num_attention_heads} heads, "
    f"head_dim = {model_attn.config.hidden_size // model_attn.config.num_attention_heads}"
)

In [ ]:
# Validate prompt/token counts once before the full model sweep
sanity_rep_id = int(hfer_ensemble_rep_ids[0])
sanity_data_name = hfer_ensemble_data_order[0]
sanity_sequence = hfer_ensemble_inputs[sanity_rep_id][sanity_data_name]["sequence"]
sanity_prompt, sanity_counts, sanity_held_out_state = _build_fixed_context_token_input_sequence(sanity_sequence)
sanity_inputs = tokenizer(
    sanity_prompt,
    add_special_tokens=hfer_tokenization_add_special_tokens,
    return_tensors="pt",
)
sanity_post_bos_token_count, sanity_token_start, sanity_final_token_id = _validate_prompt_tokenization(
    sanity_inputs.input_ids[0],
    sanity_data_name,
    sanity_rep_id,
)

if sanity_counts["total_sequence_states"] != TOTAL_SEQUENCE_STATES:
    raise ValueError(f"Expected {TOTAL_SEQUENCE_STATES} generated states.")
if sanity_held_out_state != int(sanity_sequence[CONTEXT_STATE_TOKENS]):
    raise ValueError("Held-out state must be state index 1000.")

print("Fixed-context prompt sanity check passed.")
print(
    f"  prompt uses sequence[:{CONTEXT_STATE_TOKENS}] and holds out sequence[{CONTEXT_STATE_TOKENS}] "
    f"= {sanity_held_out_state}."
)
print(
    "  {} state tokens + {} comma tokens = {} content tokens.".format(
        sanity_counts["state_tokens"],
        sanity_counts["comma_tokens"],
        sanity_counts["content_tokens"],
    )
)
print(f"  token_start={sanity_token_start}, post-BOS tokens={sanity_post_bos_token_count}.")
print(f"  final token id={sanity_final_token_id} (comma token id={COMMA_TOKEN_ID}).")

Run the full per-layer HFER sweep for each input family and realization.

In [ ]:
input_names = np.asarray(hfer_ensemble_data_order, dtype=str)
run_ids = np.asarray(hfer_ensemble_rep_ids, dtype=np.int64)

expected_shape = (EXPECTED_NUM_INPUT_FAMILIES, NUM_ENSEMBLE_RUNS, EXPECTED_NUM_LAYERS)
if (len(input_names), len(run_ids), len(layer_numbers)) != expected_shape:
    raise ValueError(
        f"Expected dimensions {expected_shape}, got "
        f"{(len(input_names), len(run_ids), len(layer_numbers))}."
    )

hfer_per_run = np.full(expected_shape, np.nan, dtype=np.float64)

device = next(model_attn.parameters()).device
# Task grid: input family x realization
tasks = [
    (input_idx, data_name, run_idx, int(rep_id))
    for input_idx, data_name in enumerate(hfer_ensemble_data_order)
    for run_idx, rep_id in enumerate(hfer_ensemble_rep_ids)
]

for input_idx, data_name, run_idx, rep_id in tqdm(tasks, desc="Fixed-context per-layer HFER"):
    entry = hfer_ensemble_inputs[int(rep_id)][data_name]
    sequence = np.asarray(entry["sequence"], dtype=int)
    input_sequence, fixed_counts, _ = _build_fixed_context_token_input_sequence(sequence)

    inputs = tokenizer(
        input_sequence,
        add_special_tokens=hfer_tokenization_add_special_tokens,
        return_tensors="pt",
    ).to(device)

    post_bos_token_count, _, final_token_id = _validate_prompt_tokenization(
        inputs.input_ids[0],
        data_name,
        rep_id,
    )
    if fixed_counts["content_tokens"] != FIXED_CONTENT_TOKENS:
        raise ValueError(f"Prompt construction count failed for {data_name}, rep_id={rep_id}.")
    if post_bos_token_count != FIXED_CONTENT_TOKENS:
        raise ValueError(f"Expected {FIXED_CONTENT_TOKENS} post-BOS tokens, got {post_bos_token_count}.")
    if final_token_id != COMMA_TOKEN_ID:
        raise ValueError(f"Expected final prompt token {COMMA_TOKEN_ID}, got {final_token_id}.")

    # Request attention and hidden-state tensors for all model layers
    with torch.no_grad():
        outputs = model_attn(
            **inputs,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
        )

    if outputs.attentions is None or outputs.hidden_states is None:
        raise ValueError("Model outputs must include attentions and hidden_states for per-layer HFER.")
    if len(outputs.attentions) != hfer_num_layers:
        raise ValueError(f"Expected {hfer_num_layers} attention layers, got {len(outputs.attentions)}.")
    if len(outputs.hidden_states) != hfer_num_layers + 1:
        raise ValueError(f"Expected {hfer_num_layers + 1} hidden-state tensors, got {len(outputs.hidden_states)}.")

    # Pair each attention layer with the same-index hidden-state layer
    hfer_bundles = compute_hfer_per_layer_from_attn_and_hidden(
        outputs.attentions,
        outputs.hidden_states,
        input_ids=inputs.input_ids[0],
        bos_token_id=tokenizer.bos_token_id,
        k_cutoff=None,
        head_weighting=hfer_head_weighting,
        drop_bos=hfer_drop_bos,
        renormalize_after_drop=hfer_renormalize_after_drop,
    )
    if len(hfer_bundles) != hfer_num_layers:
        raise ValueError(f"Expected {hfer_num_layers} HFER values, got {len(hfer_bundles)}.")

    for hfer_bundle in hfer_bundles:
        layer_idx = int(hfer_bundle["layer_idx"])
        seq_len = int(hfer_bundle["seq_len"])
        k_cutoff = int(hfer_bundle["k_cutoff"])
        attention_layer_idx = int(hfer_bundle["resolved_attention_layer_index"])
        signal_layer_idx = int(hfer_bundle["resolved_signal_layer_index"])

        if seq_len != FIXED_CONTENT_TOKENS:
            raise ValueError(
                f"Expected seq_len={FIXED_CONTENT_TOKENS}; got {seq_len} "
                f"for {data_name}, rep_id={rep_id}, layer={layer_idx + 1}."
            )
        if k_cutoff != FIXED_CONTENT_TOKENS // 2:
            raise ValueError(
                f"Expected k_cutoff={FIXED_CONTENT_TOKENS // 2}; got {k_cutoff} "
                f"for {data_name}, rep_id={rep_id}, layer={layer_idx + 1}."
            )
        if attention_layer_idx != layer_idx or signal_layer_idx != layer_idx:
            raise ValueError(
                f"Layer-pairing check failed for layer {layer_idx}: "
                f"attention={attention_layer_idx}, signal={signal_layer_idx}."
            )

        hfer_per_run[input_idx, run_idx, layer_idx] = float(hfer_bundle["hfer"])

    del outputs, inputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
expected_shape = (EXPECTED_NUM_INPUT_FAMILIES, NUM_ENSEMBLE_RUNS, EXPECTED_NUM_LAYERS)
expected_summary_shape = (EXPECTED_NUM_INPUT_FAMILIES, EXPECTED_NUM_LAYERS)

if hfer_per_run.shape != expected_shape:
    raise ValueError(f"hfer_per_run shape {hfer_per_run.shape}, expected {expected_shape}.")
if not np.array_equal(layer_numbers, np.arange(1, EXPECTED_NUM_LAYERS + 1, dtype=np.int64)):
    raise ValueError("layer_numbers must be 1..28.")
if not np.isfinite(hfer_per_run).all():
    bad = np.argwhere(~np.isfinite(hfer_per_run))[:5]
    raise ValueError(f"hfer_per_run must contain only finite values; first bad indices: {bad}.")

# Summarize across realizations
hfer_mean = np.mean(hfer_per_run, axis=1)
if NUM_ENSEMBLE_RUNS > 1:
    hfer_std = np.std(hfer_per_run, axis=1, ddof=1)
else:
    hfer_std = np.zeros_like(hfer_mean)

for array_name, array_value in {
    "hfer_mean": hfer_mean,
    "hfer_std": hfer_std,
}.items():
    if array_value.shape != expected_summary_shape:
        raise ValueError(f"{array_name} shape {array_value.shape}, expected {expected_summary_shape}.")

Save the compact arrays consumed by the companion Visualize notebook.

In [ ]:
# Save relative to this notebook's working directory
np.savez_compressed(
    OUTPUT_PATH,
    input_names=input_names,
    layer_numbers=layer_numbers,
    hfer_mean=hfer_mean,
    hfer_std=hfer_std,
)